# Agregações com PySpark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Agregacao").getOrCreate()


## Ler arquivo parquet

In [ ]:
df_video = spark.read.parquet("videos-preparados.snappy.parquet")
df_video.show(5)


## Quantidade de registros por Keyword

In [ ]:
df_video.groupBy("Keyword").count().show()


## Média de Interaction por Keyword

In [ ]:
df_video.groupBy("Keyword") \
    .agg(avg("Interaction").alias("Media Interaction")) \
    .show()


## Máximo de Interaction por Keyword

In [ ]:
df_video.groupBy("Keyword") \
    .agg(max("Interaction").alias("Rank Interactions")) \
    .orderBy(col("Rank Interactions").desc()) \
    .show()


## Média e variância de Views

In [ ]:
df_video.groupBy("Keyword") \
    .agg(
        avg("Views").alias("Media Views"),
        variance("Views").alias("Variancia Views")
    ).show()


## Média, mínimo e máximo de Views

In [ ]:
df_video.groupBy("Keyword") \
    .agg(
        round(avg("Views"), 0).alias("Media"),
        round(min("Views"), 0).alias("Minimo"),
        round(max("Views"), 0).alias("Maximo")
    ).show()


## Primeiro e último Published At

In [ ]:
df_video.groupBy("Keyword") \
    .agg(
        min("Published At").alias("Primeiro"),
        max("Published At").alias("Ultimo")
    ).show()


## Contagem normal e distinta de Title

In [ ]:
df_video.select(count("Title").alias("Total Titles")).show()

df_video.select(countDistinct("Title").alias("Titles Unicos")).show()


## Quantidade por ano

In [ ]:
df_video.groupBy("Year") \
    .count() \
    .orderBy("Year") \
    .show()


## Quantidade por ano e mês

In [ ]:
df_video.groupBy("Year", "Month") \
    .count() \
    .orderBy("Year", "Month") \
    .show()


## Média acumulativa de Likes

In [ ]:
windowSpec = Window.partitionBy("Keyword") \
                   .orderBy("Year") \
                   .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_media_acumulativa = df_video.withColumn(
    "Media Acumulativa Likes",
    avg("Likes").over(windowSpec)
)

df_media_acumulativa.select(
    "Keyword",
    "Year",
    "Likes",
    "Media Acumulativa Likes"
).show()
